In [10]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Dashboard components
import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output, State
import plotly.express as px
import base64

# System + data
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Jupyter config
JupyterDash.infer_jupyter_proxy_config()

# Import CRUD module
from AnimalShelter import AnimalShelter


###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "Ellysse22"

# Connect to DB
db = AnimalShelter(username, password)

# Load data
df = pd.DataFrame.from_records(db.read({}))

# Drop Mongo _id column
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

# Logo setup
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([

    # Header
    html.Center([
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image),
            style={'height': '250px'}
        ),
        html.H1('Grazioso Salvare Dashboard'),
        html.H4('Tommy Reid')
    ]),

    html.Hr(),

    # Filter options
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Reset', 'value': 'reset'},
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain/Wilderness', 'value': 'mountain'},
                {'label': 'Disaster/Tracking', 'value': 'disaster'}
            ],
            value='reset',
            labelStyle={'display': 'inline-block', 'margin-right': '20px'}
        )
    ]),

    html.Hr(),

    # Data table
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        page_size=10,
        sort_action="native",
        filter_action="native",
        row_selectable="single",
        selected_rows=[0],
        style_table={'overflowX': 'auto'}
    ),

    html.Br(),
    html.Hr(),

    # Graph + map side by side
    html.Div(
        className='row',
        style={'display': 'flex'},
        children=[
            html.Div(id='graph-id', style={'width': '50%'}),
            html.Div(id='map-id', style={'width': '50%'})
        ]
    )
])


#############################################
# Interaction Between Components / Controller
#############################################

# Filter callback
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    if filter_type == 'water':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever"]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'mountain':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "German Shepherd",
                "Alaskan Malamute",
                "Old English Sheepdog",
                "Siberian Husky",
                "Rottweiler"
            ]},
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == 'disaster':
        query = {
            "animal_type": "Dog",
            "breed": {"$in": [
                "Doberman Pinscher",
                "German Shepherd",
                "Golden Retriever",
                "Bloodhound",
                "Rottweiler"
            ]},
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    dff = pd.DataFrame.from_records(db.read(query))

    if '_id' in dff.columns:
        dff.drop(columns=['_id'], inplace=True)

    return dff.to_dict('records')


# Graph callback
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    if viewData is None or len(viewData) == 0:
        return [html.H4("No data available")]

    dff = pd.DataFrame.from_dict(viewData)

    if 'breed' not in dff.columns:
        return [html.H4("Breed data not available")]

    return [
        dcc.Graph(
            figure=px.pie(
                dff,
                names='breed',
                title='Breed Distribution'
            )
        )
    ]


# Table highlight callback
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Map callback
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    if viewData is None or len(viewData) == 0:
        return [html.H4("No map data available")]

    dff = pd.DataFrame.from_dict(viewData)

    if dff.empty:
        return [html.H4("No map data available")]

    # Default to first row if nothing is selected
    if index is None or len(index) == 0:
        row = 0
    else:
        row = index[0]

    # Prevent out-of-range errors after filtering
    if row >= len(dff):
        row = 0

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[dff.iloc[row, 13], dff.iloc[row, 14]],
                    children=[
                        dl.Tooltip(str(dff.iloc[row, 4])),
                        dl.Popup([
                            html.H1("Animal Name"),
                            html.P(str(dff.iloc[row, 9]))
                        ])
                    ]
                )
            ]
        )
    ]


# Run app
app.run_server(mode='inline', debug=False)

 * Running on http://127.0.0.1:8050/ (Press CTRL+C to quit)
127.0.0.1 - - [17/Apr/2026 14:20:09] "GET /_alive_64f71f35-827e-4cfe-82fb-992ed59b0e29 HTTP/1.1" 200 -


127.0.0.1 - - [17/Apr/2026 14:20:09] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:09] "GET /_dash-dependencies HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:11] "GET /_dash-layout HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:12] "GET /_dash-component-suites/dash/dash_table/async-table.js HTTP/1.1" 304 -
127.0.0.1 - - [17/Apr/2026 14:20:12] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:12] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:12] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:12] "GET /_dash-component-suites/dash/dash_table/async-highlight.js HTTP/1.1" 304 -
127.0.0.1 - - [17/Apr/2026 14:20:14] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:14] "POST /_dash-update-component HTTP/1.1" 200 -
127.0.0.1 - - [17/Apr/2026 14:20:14] "GET /_dash-component-suites/dash/dcc/async-graph.js HTTP/1.1" 304 -
127.0.0.1 - - [17/Apr/2026 14:20:15] "G